# 🤖 LLM Advisor Demo
**Tests the full flow:** XGBoost score → SHAP → clinical prompt → LM Studio (Gemma 4) → explanation + recommendations

**Prerequisites:**
- LM Studio running with Gemma 4 loaded (`http://localhost:1234`)
- `models/` folder populated (run `fall_risk_pipeline.ipynb` first)
- `pip install openai shap joblib`

---
## 1 · Setup

In [ ]:
# Install dependencies (run once; comment out after first run)
#%pip install openai shap joblib

  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.6 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 1.8 MB/s  0:00:01 eta 0:00:01
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
Using cached pydantic_core-2.46.4-cp312-cp312-macosx_11_0_arm64.whl (2.0 MB)
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
Using cached idna-3.18-py3-none-any.whl (65 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [openai]13/14 [openai]c]
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Verify LM Studio is reachable before loading the advisor
import httpx

try:
    r = httpx.get("http://localhost:1234/v1/models", timeout=5)
    models = r.json().get("data", [])
    print("LM Studio is running ✓")
    print("Loaded models:")
    for m in models:
        print(f"  → {m['id']}")
except Exception as e:
    print(f"⚠️  LM Studio not reachable: {e}")
    print("Make sure LM Studio is open and Gemma 4 is loaded.")

LM Studio is running ✓
Loaded models:
  → google/gemma-4-e4b
  → text-embedding-nomic-embed-text-v1.5


In [48]:
import importlib
import llm_advisor
importlib.reload(llm_advisor)

from llm_advisor import FallRiskAdvisor

advisor = FallRiskAdvisor(
    model_name="google/gemma-4-e4b",
    top_n_shap=6,
    temperature=0.3,
    max_tokens=4096,
)

FallRiskAdvisor ready — model: google/gemma-4-e4b @ http://localhost:1234/v1


---
## 2 · High-Risk Patient

In [28]:
high_risk_patient = {
    # Demographics
    "age": 82, "sex": "F", "bmi": 19.5,
    # Vitals / Labs
    "systolic_bp": 152, "diastolic_bp": 90, "heart_rate": 88,
    "bun": 35, "sodium": 131, "hemoglobin": 9.2,
    # Diagnoses
    "has_parkinsons": 1, "has_osteoporosis": 1, "has_diabetes": 0,
    "has_dementia": 1, "has_depression": 1, "has_hypertension": 1,
    # Medications
    "on_sedatives": 1, "on_diuretics": 1,
    "on_antihypertensives": 1, "on_anticoagulants": 0,
    # Functional
    "prior_fall": 1, "uses_assistive_device": 1,
}

In [36]:
result_high = advisor.advise(high_risk_patient)

print(f"Risk Score : {result_high['risk_score']:.1%}")
print(f"Risk Level : {result_high['risk_label']}")
print(f"\nTop SHAP Features:")
for name, val in result_high['top_shap']:
    direction = '▲' if val > 0 else '▼'
    print(f"  {direction} {name:<30} {val:+.3f}")

Risk Score : 95.3%
Risk Level : HIGH

Top SHAP Features:
  ▲ age                            +0.854
  ▲ prior_fall                     +0.411
  ▲ has_dementia                   +0.374
  ▲ has_parkinsons                 +0.363
  ▲ on_sedatives                   +0.228
  ▲ uses_assistive_device          +0.188


In [37]:
print("=" * 60)
print("RISK EXPLANATION")
print("=" * 60)
print(result_high['explanation'])
print()
print("=" * 60)
print("CLINICAL RECOMMENDATIONS")
print("=" * 60)
print(result_high['recommendations'])

RISK EXPLANATION
This 82-year-old female presents with an extremely high fall risk due to the confluence of advanced age, mobility impairment from Parkinson's disease and osteoporosis, and cognitive decline exacerbated by dementia. Contributing factors include a history of falls, polypharmacy involving sedatives, and current mobility dependence on an assistive device.

CLINICAL RECOMMENDATIONS
- Conduct a comprehensive medication review (CMR) focusing on deprescribing or adjusting sedatives/benzodiazepines to minimize CNS depression.
- Optimize gait stability and balance through physical therapy referral targeting Parkinsonian rigidity and fall mechanics.
- Address anemia (Hgb 9.2 g/dL) and elevated BUN (35 mg/dL) to improve overall endurance and reduce fall precipitants.
- Implement environmental safety checks, including bed alarms and clear pathways, given the dementia diagnosis.
- Establish a structured routine for mobility assistance while ensuring appropriate use of the assistive 

---
## 3 · Low-Risk Patient

In [49]:
low_risk_patient = {
    "age": 34, "sex": "M", "bmi": 24.1,
    "systolic_bp": 118, "diastolic_bp": 76, "heart_rate": 68,
    "bun": 14, "sodium": 140, "hemoglobin": 15.2,
    "has_parkinsons": 0, "has_osteoporosis": 0, "has_diabetes": 0,
    "has_dementia": 0, "has_depression": 0, "has_hypertension": 0,
    "on_sedatives": 0, "on_diuretics": 0,
    "on_antihypertensives": 0, "on_anticoagulants": 0,
    "prior_fall": 0, "uses_assistive_device": 0,
}

result_low = advisor.advise(low_risk_patient)

print(f"Risk Score : {result_low['risk_score']:.1%}")
print(f"Risk Level : {result_low['risk_label']}")
print(f"\nTop SHAP Features:")
for name, val in result_low['top_shap']:
    direction = '▲' if val > 0 else '▼'
    print(f"  {direction} {name:<30} {val:+.3f}")

print()
print("=" * 60)
print("RISK EXPLANATION")
print("=" * 60)
print(result_low['explanation'])
print()
print("=" * 60)
print("CLINICAL RECOMMENDATIONS")
print("=" * 60)
print(result_low['recommendations'])

Risk Score : 3.5%
Risk Level : LOW

Top SHAP Features:
  ▼ age                            -1.759
  ▼ prior_fall                     -0.612
  ▼ bmi                            -0.169
  ▼ uses_assistive_device          -0.137
  ▼ sodium                         -0.119
  ▲ diastolic_bp                   +0.106

RISK EXPLANATION
The patient presents with a low fall risk score of 3.5%, primarily due to their young age, lack of prior fall history, and stable vital signs (BP 118/76 mmHg) with normal laboratory values. All contributing factors, including BMI and sodium level, are protective against fall risk in this assessment.

CLINICAL RECOMMENDATIONS
- Maintain current activity levels and address any lifestyle factors that could influence balance.
- Counsel on proper footwear selection to ensure stability during ambulation.
- Establish baseline fall risk screening protocols for future follow-up visits.
- Review sleep hygiene and activity patterns to optimize overall physical readiness.


---
## 3 · Moderate-Risk Patient

In [50]:
moderate_risk_patient = {
    "age": 68, "sex": "F", "bmi": 26.8,
    "systolic_bp": 138, "diastolic_bp": 84, "heart_rate": 74,
    "bun": 22, "sodium": 136, "hemoglobin": 11.8,
    "has_parkinsons": 0, "has_osteoporosis": 1, "has_diabetes": 1,
    "has_dementia": 0, "has_depression": 1, "has_hypertension": 1,
    "on_sedatives": 0, "on_diuretics": 1,
    "on_antihypertensives": 1, "on_anticoagulants": 0,
    "prior_fall": 0, "uses_assistive_device": 0,
}

result_mod = advisor.advise(moderate_risk_patient)

print(f"Risk Score : {result_mod['risk_score']:.1%}")
print(f"Risk Level : {result_mod['risk_label']}")
print(f"\nTop SHAP Features:")
for name, val in result_mod['top_shap']:
    direction = '▲' if val > 0 else '▼'
    print(f"  {direction} {name:<30} {val:+.3f}")

print()
print("=" * 60)
print("RISK EXPLANATION")
print("=" * 60)
print(result_mod['explanation'])
print()
print("=" * 60)
print("CLINICAL RECOMMENDATIONS")
print("=" * 60)
print(result_mod['recommendations'])

Risk Score : 49.3%
Risk Level : MODERATE

Top SHAP Features:
  ▼ prior_fall                     -0.614
  ▲ has_depression                 +0.310
  ▲ on_diuretics                   +0.250
  ▲ age                            +0.233
  ▲ on_antihypertensives           +0.164
  ▼ on_sedatives                   -0.152

RISK EXPLANATION
The patient received a MODERATE fall risk score primarily due to the combination of advanced age and polypharmacy involving diuretics and antihypertensives, which increases the risk of orthostatic hypotension or electrolyte imbalance. This physiological vulnerability is compounded by a documented history of depression, which itself is a known independent risk factor for falls.

CLINICAL RECOMMENDATIONS
- Conduct comprehensive medication reconciliation to assess potential side effects of diuretics and antihypertensives.
- Screen for depression using a validated tool (e.g., PHQ-9) and initiate referral to mental health services.
- Initiate a structured physical t

In [46]:
print(result_mod)

{'risk_score': 0.49301478266716003, 'risk_label': 'MODERATE', 'top_shap': [('prior_fall', -0.6140528917312622), ('has_depression', 0.3102492392063141), ('on_diuretics', 0.24961020052433014), ('age', 0.2329980731010437), ('on_antihypertensives', 0.16386650502681732), ('on_sedatives', -0.1522856056690216)], 'explanation': '', 'recommendations': '', 'full_response': ''}


---
## 5 · Batch Scoring

In [34]:
import pandas as pd

# Score only (no LLM call) — useful for batch screening
patients = [high_risk_patient, moderate_risk_patient, low_risk_patient]
labels   = ["High Risk", "Moderate Risk", "Low Risk"]

rows = []
for p, label in zip(patients, labels):
    score, top_shap = advisor.score(p)
    rows.append({
        "Patient":      label,
        "Age":          p["age"],
        "Risk Score":   f"{score:.1%}",
        "Risk Level":   "HIGH" if score >= 0.65 else "MODERATE" if score >= 0.35 else "LOW",
        "Top Factor":   top_shap[0][0] if top_shap else "",
    })

pd.DataFrame(rows)

,Patient,Age,Risk Score,Risk Level,Top Factor
0,High Risk,82,95.3%,HIGH,age
1,Moderate Risk,68,49.3%,MODERATE,prior_fall
2,Low Risk,34,3.5%,LOW,age


---
## 6 · Inspect the Prompt (optional)
Useful for debugging or tweaking the LLM prompt.

In [35]:
from llm_advisor import _build_prompt

score, top_shap = advisor.score(high_risk_patient)
prompt = _build_prompt(high_risk_patient, score, top_shap)
print(prompt)

You are a clinical decision support assistant specializing in fall prevention.

## Patient Summary
- Age: 82 | Sex: F | BMI: 19.5
- Diagnoses: Parkinsons, Osteoporosis, Dementia, Depression, Hypertension
- Active medications: Sedatives, Diuretics, Antihypertensives
- Prior fall history: Yes
- Uses assistive device: Yes
- Key labs: Na 131 mEq/L | Hgb 9.2 g/dL | BUN 35 mg/dL
- Vitals: BP 152/90 mmHg | HR 88 bpm

## AI Risk Assessment
- Fall risk score: 95.3%
- Risk level: **HIGH**

## Top Contributing Factors (from SHAP analysis)
  - Patient age: increases risk (impact score: 0.854)
  - History of prior fall: increases risk (impact score: 0.411)
  - Dementia: increases risk (impact score: 0.374)
  - Parkinson's disease: increases risk (impact score: 0.363)
  - Currently on sedatives/benzodiazepines: increases risk (impact score: 0.228)
  - Uses assistive device (walker/cane): increases risk (impact score: 0.188)

## Your Task
Respond with ONLY a valid JSON object — no markdown, no code f

---
## ✅ Summary

| Component | Detail |
|-----------|--------|
| **Model** | XGBoost loaded from `models/fall_risk_xgb.joblib` |
| **Explainer** | SHAP TreeExplainer — top 6 features per patient |
| **LLM** | Gemma 4 via LM Studio (OpenAI-compatible API) |
| **Output** | Risk score + level + explanation + recommendations |

**Next step:** Import `FallRiskAdvisor` into `app.py` (Streamlit) and wire it to the patient input form.